In [ ]:
# !pip install chromadb pypdf google-genai python-dotenv

import os
import time
import chromadb
from pypdf import PdfReader
from google import genai
import dotenv

dotenv.load_dotenv()

# --- SYSTEM CONFIGURATION ---
# Replace with your actual key or ensure it is in your .env file
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY")
ai_client = genai.Client()

TARGET_DOCUMENT = "YOUR_PDF_FILE_HERE.pdf"
VECTOR_MODEL = "gemini-embedding-001"
CHAT_MODEL = "gemini-2.5-flash"

# Adjusted segmentation parameters
MAX_CHARS_PER_SEGMENT = 600
SEGMENT_OVERLAP = 150

In [ ]:
def parse_and_segment(filepath: str, length: int, overlap: int) -> list[str]:
    """Extracts text and segments it securely to prevent logic loops and ghost PDFs."""
    # 1. Prevent the infinite loop math error
    if overlap >= length:
        raise ValueError("Configuration Error: Overlap must be strictly less than segment length.")
        
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"Missing file: {filepath}")

    doc_reader = PdfReader(filepath)
    # Different extraction syntax (list comprehension)
    extracted_content = "".join([page.extract_text() or "" for page in doc_reader.pages])

    # 2. Prevent the 'Scanned PDF' silent failure
    if not extracted_content.strip():
        raise ValueError("Extraction Failed: The PDF contains no readable text (likely a scanned image).")

    segments = []
    cursor = 0
    while cursor < len(extracted_content):
        boundary = cursor + length
        segments.append(extracted_content[cursor:boundary])
        cursor += (length - overlap)
        
    return segments

def fetch_vector(text_data: str) -> list[float]:
    """Wraps the API call in a try-except block to catch network/auth errors."""
    try:
        response = ai_client.models.embed_content(
            model=VECTOR_MODEL,
            contents=text_data
        )
        return response.embeddings[0].values
    except Exception as e:
        print(f"-> API Error during embedding: {e}")
        return []

In [ ]:
print("Initializing document segmentation...")
try:
    text_segments = parse_and_segment(TARGET_DOCUMENT, MAX_CHARS_PER_SEGMENT, SEGMENT_OVERLAP)
    print(f"Generated {len(text_segments)} text segments.")
except Exception as e:
    print(f"Critical Error during parsing: {e}")
    text_segments = []

if text_segments:
    # Changed database directory and collection name
    db_client = chromadb.PersistentClient(path="./local_vector_store")
    knowledge_base = db_client.get_or_create_collection(name="pdf_knowledge_base")

    print("Uploading to Vector Database (this may take a moment)...")
    for index, segment in enumerate(text_segments):
        vec = fetch_vector(segment)
        
        if not vec:
            print(f"Skipping segment {index} due to vectorization failure.")
            continue
            
        # Unique ID generation
        doc_id = f"doc_{os.path.basename(TARGET_DOCUMENT)}_seg_{index}"
        
        try:
            knowledge_base.upsert(
                ids=[doc_id],
                embeddings=[vec],
                documents=[segment]
            )
        except Exception as e:
            print(f"-> Database upload failed for segment {index}: {e}")
            
        # 3. Add a small delay to avoid triggering API Rate Limits (HTTP 429)
        time.sleep(0.5) 

    print("Database populated and ready.")

In [ ]:
def generate_rag_response(user_input: str) -> str:
    """Retrieves context and asks the LLM, heavily guarded with try-except blocks."""
    try:
        input_vector = fetch_vector(user_input)
        if not input_vector:
            return "System Error: Could not vectorize your query."

        # Changed n_results to 4 to give the model slightly more context
        search_results = knowledge_base.query(
            query_embeddings=[input_vector],
            n_results=4  
        )
        
        found_docs = search_results.get("documents", [[]])[0]
        context_block = "\n\n---\n\n".join(found_docs)
        
        # Reformatted prompt structure
        strict_prompt = (
            f"You are an AI assistant. Answer the user's question STRICTLY based on the provided context.\n\n"
            f"CONTEXT:\n{context_block}\n\n"
            f"USER QUESTION: {user_input}"
        )
        
        reply = ai_client.models.generate_content(
            model=CHAT_MODEL,
            contents=strict_prompt,
        )
        return reply.text
        
    except Exception as api_err:
        return f"Generation Error: {api_err}"

# Execution loop
print("--- RAG Assistant Online ---")
while True:
    question = input("Ask a question (or type 'quit'): ").strip()
    
    if question.lower() in ['quit', 'exit']:
        print("Shutting down...")
        break
    if not question:
        continue
        
    print("\nThinking...")
    answer = generate_rag_response(question)
    print(f"\nAI: {answer}\n")
    print("-" * 50)